# Phase 4, Stage 3b: Chi-Squared Tests

## What this test does

The KS tests (Stage 3a) looked at the full continuous `capped_cpl` distribution. This test looks at the **categorical composition** — out of all moves in a rating band, what proportion are Inaccuracies / Minor Errors / Major Errors / Blunders, and does that mix change across the four time pressure bins?

For each rating band, we build a 4x4 contingency table (rows = time pressure bin, columns = error category) and run a chi-squared test of independence.

- **H0 (null hypothesis):** error category proportions are the same across all four time pressure bins (i.e. time pressure and error severity category are independent).
- **H1 (alternate hypothesis):** error category proportions differ across at least one time pressure bin.

As with every test in this project, **failing to reject H0 is a perfectly valid, useful outcome** — it would tell us that whatever shape changes the KS test picked up, they aren't large enough to shift moves *between* the four broad error categories. That's a meaningful (if less dramatic) finding, not a failed analysis.

## Multiple comparisons correction

One chi-squared test per rating band = **5 tests total**.

**Bonferroni-corrected α = 0.05 / 5 = 0.01**

## Effect size: Cramer's V

Like the KS D statistic, chi-squared's test statistic itself is sample-size dependent — with ~150,000-230,000 moves per band, almost any real difference will be "significant". **Cramer's V** rescales the chi-squared statistic to a 0-1 range (for a 4x4 table, V=1 would mean a perfect, complete shift) and is the categorical analogue of the D statistic — it tells us how *big* the compositional shift is, independent of sample size.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

ALPHA_BONFERRONI = 0.05 / 5  # 0.01, for the 5 chi-squared tests

RATING_BAND_LABELS = {
    1: 'Novice (<1000)',
    2: 'Intermediate (1000-1499)',
    3: 'Club Player (1500-1999)',
    4: 'Advanced (2000-2299)',
    5: 'Expert/Master (2300+)',
}

ERROR_CATEGORY_LABELS = {
    1: 'Inaccuracy',
    2: 'Minor Error',
    3: 'Major Error',
    4: 'Blunder',
}

df = pd.read_csv('../../data/processed/analysed_moves.csv')
print(f'Loaded {len(df):,} rows')
print(f'Bonferroni-corrected alpha for chi-squared tests: {ALPHA_BONFERRONI}')

In [ ]:
def cramers_v(chi2, n, table_shape):
    r, c = table_shape
    return np.sqrt((chi2 / n) / (min(r, c) - 1))

results = []
contingency_tables = {}

for rating_band in sorted(RATING_BAND_LABELS):
    cell = df[df['rating_band'] == rating_band]

    # rows = time_pressure_bin (1-4), columns = error_category (1-4)
    table = pd.crosstab(cell['time_pressure_bin'], cell['error_category'])
    contingency_tables[rating_band] = table

    chi2, p, dof, expected = stats.chi2_contingency(table)
    n = table.values.sum()
    v = cramers_v(chi2, n, table.shape)

    results.append({
        'rating_band': rating_band,
        'rating_band_label': RATING_BAND_LABELS[rating_band],
        'n': n,
        'chi2': chi2,
        'dof': dof,
        'p_value': p,
        'cramers_v': v,
        'significant_bonferroni': p < ALPHA_BONFERRONI,
    })

chi2_results = pd.DataFrame(results)
chi2_results

## Save results

In [ ]:
chi2_results.to_csv('../results/chi_squared_results.csv', index=False)
print('Saved to ../results/chi_squared_results.csv')